# 04 — Petit essai : télécharger 100 jumelages depuis Wikidata

**Objectif :** faire un **scrapping manuel limité** (1 requête, ~100 lignes) sans lancer une sync mondiale complète.

**Prérequis :**
- Connexion Internet
- `pip install -r requirements.txt`
- Idéalement définir ton email (règle Wikidata) :
  ```bash
  set WIKIDATA_CONTACT_EMAIL=ton.email@etu.fr
  ```

> **Note :** 1 ligne = 1 **jumelage** (paire de villes), pas 1 ville seule.

## Option A — Script dédié (recommandé)

Le script [`scripts/sync_sample.py`](../scripts/sync_sample.py) :
- Télécharge **100 jumelages** par défaut (`--limit` modifiable)
- **Remplace** le CSV existant (efface les données test)
- Regénère `graphe_jumelages.json` si le CSV change
- Écrit `metadata.json` avec `"sync_mode": "sample"`

Dans un terminal à la **racine du projet** :

```bash
python scripts/sync_sample.py
python scripts/build_stats.py
```

In [ ]:
# Option B — Lancer l'essai depuis le notebook (equivalent au script)
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("..").resolve()
LIMIT = 100  # ← change ce nombre pour tester (ex: 20, 50)

# Decommente et adapte si Wikidata doit t'identifier :
# os.environ["WIKIDATA_CONTACT_EMAIL"] = "ton.email@etu.fr"

cmd = [sys.executable, str(ROOT / "scripts" / "sync_sample.py"), "--limit", str(LIMIT)]
print("Commande:", " ".join(cmd))
print("(Decommente la ligne subprocess.run ci-dessous pour executer)")

# subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
# Apres le sync : regenerer les stats
# subprocess.run([sys.executable, str(ROOT / "scripts" / "build_stats.py")], cwd=ROOT, check=True)

## Option C — Appel Python direct (sans subprocess)

Utile pour comprendre ce que fait le script sous le capot.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "scripts"))
from sync_wikidata import run_sync

LIMIT = 20  # petit test rapide

# Decommente pour executer (requiert acces Wikidata) :
# code = run_sync(full=True, limit=LIMIT, replace=True, sync_mode="sample")
# print(f"Code retour: {code} (1 = fichiers modifies)")

## Vérifier le résultat

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("..") / "data" / "processed"
CSV = DATA / "jumelages.csv"
META = DATA / "metadata.json"

if CSV.exists():
    df = pd.read_csv(CSV)
    print(f"Jumelages dans le CSV : {len(df)}")
    display(df.head(10))
else:
    print("Pas encore de CSV — lance sync_sample.py d'abord.")

if META.exists():
    meta = json.loads(META.read_text(encoding="utf-8"))
    print()
    print("Metadata :")
    for k in ["sync_mode", "rows_fetched", "rows_total", "last_run"]:
        print(f"  {k}: {meta.get(k)}")

## Bonnes pratiques (parcimonie)

| À faire | À éviter |
|---------|----------|
| 1 essai à `--limit 100` | Relancer 10× de suite |
| Attendre si erreur **429** | Marteler le serveur Wikidata |
| Renseigner `WIKIDATA_CONTACT_EMAIL` | User-Agent anonyme |
| `--full` seulement pour la prod | Confondre essai et sync hebdo CI |

## Variantes utiles

```bash
# 50 jumelages seulement
python scripts/sync_sample.py --limit 50

# Ajouter au CSV existant au lieu de remplacer
python scripts/sync_sample.py --limit 20 --merge

# Sync complete (production) — long, beaucoup de requetes
python scripts/sync_wikidata.py --full
```

**Suite :** notebook **03** pour explorer les données téléchargées.